## Efek Domino Konflik Geopolitik Iran pada Diskursus Industri Plastik Indonesia
## Analisis Topic Modeling pada Pemberitaan Media Online

**Pipeline:** Preprocessing → IndoBERT Embedding → BERTopic (UMAP + HDBSCAN) → Evaluasi & Visualisasi

---
## 1. Instalasi Library

In [2]:
%%capture
!pip install bertopic
!pip install transformers
!pip install sentence-transformers
!pip install PySastrawi
!pip install umap-learn
!pip install hdbscan
!pip install plotly
!pip install scikit-learn

---
## 2. Import Library

In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Embedding
from sentence_transformers import SentenceTransformer

# Topic Modeling
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Visualisasi
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

print('Semua library berhasil diimport!')

Semua library berhasil diimport!


---
## 3. Load Data

In [4]:
# Upload file CSV ke Colab terlebih dahulu
from google.colab import files
uploaded = files.upload()

Saving berita_plastik (1).csv to berita_plastik (1).csv


In [5]:
df = pd.read_csv('berita_plastik (1).csv')
print(f'Total data: {df.shape[0]} berita')
print(f'Kolom: {df.columns.tolist()}')
df.head()

Total data: 1541 berita
Kolom: ['title', 'publish_date', 'author', 'content', 'keyword', 'category', 'source', 'link']


,title,publish_date,author,content,keyword,category,source,link
0,"Di Balik Mahalnya Plastik, Waspadai Risiko Kes...",2026-04-08 17:11:00,Oleh :Rizkya Fajarani Bahar,"Jakarta, VIVA – Selain minyak mentah yang menj...",harga plastik konflik iran,Lifestyle,viva.co.id,https://www.viva.co.id/gaya-hidup/1890595-di-b...
1,Konflik AS-Iran Picu Kenaikan Harga Plastik hi...,2026-04-08 16:41:00,Oleh :Abdul Aziz Masindo,"Jakarta, VIVA – Kenaikan harga plastik yang te...",harga plastik konflik iran,Bisnis,viva.co.id,https://www.viva.co.id/bisnis/1890590-konflik-...
2,5 Makanan yang Sebaiknya Tidak Pernah Disimpan...,2026-04-11 11:15:00,Afif Khoirul Muttaqin,KOMPAS.COM - Penggunaan wadah plastik memang t...,harga biji plastik resin 2026,Kompas.com/Food/Tips Kuliner,kompas.com,https://www.kompas.com/food/read/2026/04/11/11...
3,"Strategi Pemerintah Lawan El Nino, Isi Penuh B...",2026-04-11 11:18:00,"Aisyah Sekar Ayu Maharani,Hilda B Alexander","JAKARTA, KOMPAS.com - Menteri Pekerjaan Umum (...",harga biji plastik resin 2026,Kompas.com/Properti/Berita,kompas.com,https://www.kompas.com/properti/read/2026/04/1...
4,"Promo Superindo Hari Ini 11 April 2026, Harga ...",2026-04-11 11:30:00,Tim Kompas.com,KOMPAS.com - Superindo Supermarket kembali men...,harga biji plastik resin 2026,Kompas.com/Jawa Tengah,kompas.com,https://www.kompas.com/jawa-tengah/read/2026/0...


---
## 4. Filtering Berita Relevan

Filter berita yang benar-benar relevan dengan topik: plastik + konteks Iran/geopolitik/harga/industri.

In [6]:
# Keyword filter — plastik harus ada, plus minimal satu konteks relevan
keyword_plastik = r'plastik|resin|polimer|biji plastik'
keyword_konteks = r'iran|timur tengah|geopolitik|harga|industri|rantai pasok|impor|ekspor|konflik|minyak|petrokimia|bbm|produksi|pabrik|daur ulang'

# Gabungkan title + content untuk filtering
df['text_gabung'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

mask = (
    df['text_gabung'].str.contains(keyword_plastik, case=False, na=False) &
    df['text_gabung'].str.contains(keyword_konteks, case=False, na=False)
)

df_filtered = df[mask].reset_index(drop=True)
print(f'Berita relevan: {len(df_filtered)} dari {len(df)} total')
df_filtered[['title', 'publish_date', 'source']].head(10)

Berita relevan: 142 dari 1541 total


,title,publish_date,source
0,"Di Balik Mahalnya Plastik, Waspadai Risiko Kes...",2026-04-08 17:11:00,viva.co.id
1,Konflik AS-Iran Picu Kenaikan Harga Plastik hi...,2026-04-08 16:41:00,viva.co.id
2,5 Makanan yang Sebaiknya Tidak Pernah Disimpan...,2026-04-11 11:15:00,kompas.com
3,Akumindo Sebut Dampak Kenaikan Harga Plastik k...,2026-04-08 15:03:00,bisnis.com
4,"Harga Plastik Naik hingga 60%, Pemkot Surabaya...",2026-04-07 14:39:00,bisnis.com
5,Harga Plastik Naik Imbas Perang Iran vs Israel...,2026-04-10 00:16:00,bisnis.com
6,"""Babak Belur"" UMKM, Terpukul Lonjakan Harga Pl...",2026-04-11 12:00:00,kompas.com
7,5 Makanan yang Sebaiknya Tidak Pernah Disimpan...,2026-04-11 11:15:00,kompas.com
8,"Harga Plastik Menonjak, Penjual Es untuk Nelay...",2026-04-11 10:22:00,kompas.com
9,Harga Plastik di Surabaya Melejit 40% Jadi Efe...,2026-04-03 16:00:23,cnnindonesia.com


---
## 5. Preprocessing Teks

In [7]:
# Inisialisasi Sastrawi
stop_factory = StopWordRemoverFactory()
stemmer_factory = StemmerFactory()

stopwords = stop_factory.get_stop_words()
stemmer = stemmer_factory.create_stemmer()

# Tambah custom stopwords domain
custom_stopwords = [
    'kata', 'ujar', 'ungkap', 'jelas', 'tutur', 'sebut', 'terang',
    'dilansir', 'dikutip', 'menurut', 'berdasarkan', 'bahwa', 'tersebut',
    'jakarta', 'indonesia', 'com', 'www', 'http', 'https', 'foto', 'baca',
    'juga', 'lebih', 'saat', 'tahun', 'persen', 'rp', 'rb', 'jt'
]
stopwords = list(set(stopwords + custom_stopwords))
print(f'Total stopwords: {len(stopwords)}')

Total stopwords: 826


In [8]:
def clean_text(text):
    """Cleaning dasar: hapus HTML, URL, angka, karakter spesial"""
    if not isinstance(text, str):
        return ''
    # Hapus HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Hapus URL
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Hapus angka
    text = re.sub(r'\d+', ' ', text)
    # Hapus karakter spesial, sisakan huruf dan spasi
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase
    text = text.lower()
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text, stopwords):
    """Hapus stopwords"""
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
    return ' '.join(tokens)

def stem_text(text, stemmer):
    """Stemming Bahasa Indonesia"""
    return stemmer.stem(text)

print('Fungsi preprocessing siap.')

Fungsi preprocessing siap.


In [9]:
# Gunakan content sebagai dokumen utama, fallback ke title jika content kosong
df_filtered['doc_raw'] = df_filtered['content'].fillna('').apply(
    lambda x: x if len(x) > 50 else df_filtered.loc[df_filtered['content'] == x, 'title'].values[0]
)

# Gunakan title + content untuk dokumen
df_filtered['doc_raw'] = (df_filtered['title'].fillna('') + '. ' + df_filtered['content'].fillna(''))

# Step 1: Clean
print('Step 1: Cleaning...')
df_filtered['doc_clean'] = df_filtered['doc_raw'].apply(clean_text)

# Step 2: Remove stopwords
print('Step 2: Remove stopwords...')
df_filtered['doc_nostop'] = df_filtered['doc_clean'].apply(
    lambda x: remove_stopwords(x, stopwords)
)

# Tanpa stemming (lebih cepat, BERTopic sudah pakai semantic embedding)
docs = df_filtered['doc_nostop'].tolist()

# Hapus dokumen kosong
docs = [d for d in docs if len(d.strip()) > 10]

print(f'\nTotal dokumen siap diproses: {len(docs)}')
print('\nContoh dokumen setelah preprocessing:')
print(docs[0][:300])

Step 1: Cleaning...
Step 2: Remove stopwords...

Total dokumen siap diproses: 142

Contoh dokumen setelah preprocessing:
mahalnya plastik waspadai risiko kesehatan wadah makan pakai viva minyak mentah sorotan tengah konflik amerika serikat iran harga plastik perhatian geger pedagang tanah air ahli menjabarkan dampak mahalnya harga plastik berimbas pelaku industri konsumen scroll yuk advertisement gulir kenaikan harga 


---
## 6. Embedding dengan IndoBERT

Menggunakan `indobenchmark/indobert-base-p1` via SentenceTransformer.

In [10]:
# Load IndoBERT
print('Loading IndoBERT...')
embedding_model = SentenceTransformer('indobenchmark/indobert-base-p1')
print('Model loaded!')

# Generate embeddings
print('Generating embeddings (ini butuh beberapa menit)...')
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=32)
print(f'\nEmbedding shape: {embeddings.shape}')

Loading IndoBERT...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!
Generating embeddings (ini butuh beberapa menit)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding shape: (142, 768)


---
## 7. Konfigurasi & Training BERTopic

In [11]:
# UMAP untuk dimensionality reduction
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN untuk clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer dengan stopwords Bahasa Indonesia
vectorizer_model = CountVectorizer(
    stop_words=stopwords,
    min_df=2,
    ngram_range=(1, 2)  # unigram + bigram
)

# Inisialisasi BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    nr_topics='auto',  # biarkan model tentukan jumlah topik
    verbose=True
)

print('Konfigurasi BERTopic selesai.')

Konfigurasi BERTopic selesai.


In [12]:
# Training BERTopic
print('Training BERTopic...')
topics, probs = topic_model.fit_transform(docs, embeddings)

print(f'\nJumlah topik ditemukan: {len(set(topics)) - 1} (exclude outlier -1)')
print(f'Jumlah outlier (-1): {topics.count(-1)}')

2026-04-16 12:59:42,817 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic...


2026-04-16 12:59:52,235 - BERTopic - Dimensionality - Completed ✓
2026-04-16 12:59:52,236 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-16 12:59:52,247 - BERTopic - Cluster - Completed ✓
2026-04-16 12:59:52,248 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-16 12:59:52,326 - BERTopic - Representation - Completed ✓
2026-04-16 12:59:52,327 - BERTopic - Topic reduction - Reducing number of topics
2026-04-16 12:59:52,336 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-16 12:59:52,400 - BERTopic - Representation - Completed ✓
2026-04-16 12:59:52,402 - BERTopic - Topic reduction - Reduced number of topics from 9 to 9



Jumlah topik ditemukan: 8 (exclude outlier -1)
Jumlah outlier (-1): 0


---
## 8. Eksplorasi Hasil Topik

In [13]:
# Info semua topik
topic_info = topic_model.get_topic_info()
print(f'Total topik (termasuk outlier): {len(topic_info)}')
topic_info

Total topik (termasuk outlier): 9


,Topic,Count,Name,Representation,Representative_Docs
0,0,27,0_harga_plastik_harga plastik_umkm,"[harga, plastik, harga plastik, umkm, maman, p...",[babak belur umkm terpukul lonjakan harga plas...
1,1,25,1_bahan_tengah_plastik_harga,"[bahan, tengah, plastik, harga, industri, baha...",[selat hormuz gonjang ganjing harga plastik ma...
2,2,19,2_penjual_bungkus_kompas_harga,"[penjual, bungkus, kompas, harga, terpaksa, ba...",[harga plastik menonjak penjual nelayan terpak...
3,3,19,3_harga_plastik_pedagang_harga plastik,"[harga, plastik, pedagang, harga plastik, kena...",[harga plastik pemkab bangkalan imbau warga pa...
4,4,14,4_wadah_makanan_cepat_buah,"[wadah, makanan, cepat, buah, plastik, penyimp...",[makanan disimpan wadah plastik kompas penggun...
5,5,13,5_umkm_harga_usaha_plastik,"[umkm, harga, usaha, plastik, pelaku, pelaku u...",[harga plastik tajam umkm kuliner terdampak ko...
6,6,11,6_bahan_bahan baku_baku_industri,"[bahan, bahan baku, baku, industri, plastik, h...",[harga plastik umkm tertekan industri beralih ...
7,7,7,7_bawa_tas_belanja_kantong,"[bawa, tas, belanja, kantong, ulang, plastik, ...",[harga plastik travelling hemat plastik kompas...
8,8,7,8_sampah_kawasan_nasional_gunung,"[sampah, kawasan, nasional, gunung, ekosistem,...",[ton sampah diangkut kawasan bromo surabaya cn...


In [14]:
# Top words per topik
print('=== TOP WORDS PER TOPIK ===')
for topic_id in topic_info['Topic'].tolist():
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    word_list = [w[0] for w in words[:10]]
    count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
    print(f'\nTopik {topic_id} ({count} dokumen):')
    print(' | '.join(word_list))

=== TOP WORDS PER TOPIK ===

Topik 0 (27 dokumen):
harga | plastik | harga plastik | umkm | maman | pelaku umkm | pelaku | pemerintah | kenaikan | beras

Topik 1 (25 dokumen):
bahan | tengah | plastik | harga | industri | bahan baku | baku | pasokan | timur tengah | timur

Topik 2 (19 dokumen):
penjual | bungkus | kompas | harga | terpaksa | batang | plastik | harga plastik | pagi | menaikan

Topik 3 (19 dokumen):
harga | plastik | pedagang | harga plastik | kenaikan | kemasan | non | kenaikan harga | kompas | usaha

Topik 4 (14 dokumen):
wadah | makanan | cepat | buah | plastik | penyimpanan | mudah | kaca | saran | ilustrasi

Topik 5 (13 dokumen):
umkm | harga | usaha | plastik | pelaku | pelaku usaha | kemasan | harga plastik | kenaikan | kenaikan harga

Topik 6 (11 dokumen):
bahan | bahan baku | baku | industri | plastik | harga | pasokan | global | tengah | tekanan

Topik 7 (7 dokumen):
bawa | tas | belanja | kantong | ulang | plastik | tas belanja | botol | sedotan | makan

Topik

In [15]:
# Dokumen representatif per topik
print('=== DOKUMEN REPRESENTATIF PER TOPIK ===')
rep_docs = topic_model.get_representative_docs()
for topic_id, doc_list in rep_docs.items():
    if topic_id == -1:
        continue
    print(f'\n--- Topik {topic_id} ---')
    for d in doc_list[:2]:
        print(f'  > {d[:200]}...')

=== DOKUMEN REPRESENTATIF PER TOPIK ===

--- Topik 0 ---
  > babak belur umkm terpukul lonjakan harga plastik kompas usaha mikro menengah umkm babak belur dipukul mahalnya harga plastik waktu lonjakan harga plastik pelaku umkm tertekan laba berkurang pedagang b...
  > babak belur umkm terpukul lonjakan harga plastik kompas usaha mikro menengah umkm babak belur dipukul mahalnya harga plastik waktu lonjakan harga plastik pelaku umkm tertekan laba berkurang pedagang b...

--- Topik 1 ---
  > selat hormuz gonjang ganjing harga plastik mahal cnn harga plastik dilaporkan mengalami kenaikan waktu pedagang pasaran merasakan lonjakan pantauan cnnindonesia wilayah lenteng agung selatan senin ken...
  > selat hormuz gonjang ganjing harga plastik mahal cnn harga plastik dilaporkan mengalami kenaikan waktu pedagang pasaran merasakan lonjakan pantauan cnnindonesia wilayah lenteng agung selatan senin ken...

--- Topik 2 ---
  > harga plastik menonjak penjual nelayan terpaksa naikkan harga labuan bajo

---
## 9. Labeling Topik Manual

Setelah melihat top words dan dokumen representatif, beri label pada tiap topik secara manual.

In [16]:
topic_labels = {
    0: 'Kenaikan Harga Plastik & Tekanan UMKM',
    1: 'Gangguan Rantai Pasok & Selat Hormuz',
    2: 'Dampak Pedagang Lokal & Nelayan',
    3: 'Respons Pedagang & Kemasan Alternatif',
    4: 'Risiko Kesehatan Wadah Plastik',
    5: 'Tekanan UMKM Sektor Kuliner',
    6: 'Gangguan Industri & Bahan Baku',
    7: 'Solusi Pengurangan Penggunaan Plastik',
    8: 'Isu Sampah Plastik & Lingkungan'
}

# Update label di model
topic_model.set_topic_labels(topic_labels)
topic_info = topic_model.get_topic_info()
print('Label topik berhasil diset!')
print('Catatan: sesuaikan topic_labels dengan hasil aktual di atas.')

Label topik berhasil diset!
Catatan: sesuaikan topic_labels dengan hasil aktual di atas.


---
## 10. Visualisasi

In [17]:
# 1. Bar chart frekuensi topik
fig1 = topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=8,
    title='Top Words per Topik'
)
fig1.show()

In [18]:
# 2. Intertopic distance map
fig2 = topic_model.visualize_topics(title='Peta Jarak Antar Topik')
fig2.show()

In [19]:
fig_heatmap = topic_model.visualize_heatmap(title='Heatmap Kemiripan Antar Topik')

# Update layout agar colorbar tidak menutupi label miring
fig_heatmap.update_layout(
    margin=dict(r=150, b=100), # Tambah margin kanan dan bawah
    coloraxis_colorbar=dict(
        x=1.2, # Geser keterangan similarity lebih ke kanan
        title="Similarity"
    )
)
fig_heatmap.update_xaxes(tickangle=45)
fig_heatmap.show()

In [20]:
# 4. Document distribution (scatter UMAP)
fig4 = topic_model.visualize_documents(
    docs,
    embeddings=embeddings,
    title='Distribusi Dokumen per Topik'
)
fig4.show()

In [21]:
# 5. Topic frequency bar chart custom
topic_freq = topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']].sort_values('Count', ascending=False)

fig5 = px.bar(
    topic_freq,
    x='Count',
    y='Name',
    orientation='h',
    title='Frekuensi Dokumen per Topik',
    color='Count',
    color_continuous_scale='Blues'
)
fig5.update_layout(yaxis={'categoryorder': 'total ascending'})
fig5.show()

---
## 12. Export Hasil

In [22]:
# Tambahkan hasil topik ke dataframe
df_filtered['topic'] = topics
df_filtered['topic_prob'] = probs

# Gabungkan dengan label topik
df_filtered['topic_label'] = df_filtered['topic'].map(topic_labels).fillna('Outlier / Tidak Relevan')

# Export
df_filtered[['title', 'publish_date', 'source', 'topic', 'topic_prob', 'topic_label']].to_csv(
    'hasil_topic_modeling.csv', index=False
)

print('File hasil_topic_modeling.csv berhasil disimpan!')
df_filtered[['title', 'publish_date', 'topic', 'topic_label']].head(10)

File hasil_topic_modeling.csv berhasil disimpan!


,title,publish_date,topic,topic_label
0,"Di Balik Mahalnya Plastik, Waspadai Risiko Kes...",2026-04-08 17:11:00,4,Risiko Kesehatan Wadah Plastik
1,Konflik AS-Iran Picu Kenaikan Harga Plastik hi...,2026-04-08 16:41:00,1,Gangguan Rantai Pasok & Selat Hormuz
2,5 Makanan yang Sebaiknya Tidak Pernah Disimpan...,2026-04-11 11:15:00,4,Risiko Kesehatan Wadah Plastik
3,Akumindo Sebut Dampak Kenaikan Harga Plastik k...,2026-04-08 15:03:00,5,Tekanan UMKM Sektor Kuliner
4,"Harga Plastik Naik hingga 60%, Pemkot Surabaya...",2026-04-07 14:39:00,5,Tekanan UMKM Sektor Kuliner
5,Harga Plastik Naik Imbas Perang Iran vs Israel...,2026-04-10 00:16:00,5,Tekanan UMKM Sektor Kuliner
6,"""Babak Belur"" UMKM, Terpukul Lonjakan Harga Pl...",2026-04-11 12:00:00,0,Kenaikan Harga Plastik & Tekanan UMKM
7,5 Makanan yang Sebaiknya Tidak Pernah Disimpan...,2026-04-11 11:15:00,4,Risiko Kesehatan Wadah Plastik
8,"Harga Plastik Menonjak, Penjual Es untuk Nelay...",2026-04-11 10:22:00,2,Dampak Pedagang Lokal & Nelayan
9,Harga Plastik di Surabaya Melejit 40% Jadi Efe...,2026-04-03 16:00:23,0,Kenaikan Harga Plastik & Tekanan UMKM


In [23]:
# Ringkasan akhir
print('=== RINGKASAN HASIL TOPIC MODELING ===')
print(f'Total berita dianalisis   : {len(docs)}')
print(f'Jumlah topik ditemukan    : {len(set(topics)) - 1}')
print(f'Jumlah outlier            : {list(topics).count(-1)}')
print(f'Persentase outlier        : {list(topics).count(-1)/len(topics)*100:.1f}%')
print()
print('Distribusi topik:')
print(topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']].to_string(index=False))

=== RINGKASAN HASIL TOPIC MODELING ===
Total berita dianalisis   : 142
Jumlah topik ditemukan    : 8
Jumlah outlier            : 0
Persentase outlier        : 0.0%

Distribusi topik:
 Topic  Count                                   Name
     0     27     0_harga_plastik_harga plastik_umkm
     1     25           1_bahan_tengah_plastik_harga
     2     19         2_penjual_bungkus_kompas_harga
     3     19 3_harga_plastik_pedagang_harga plastik
     4     14             4_wadah_makanan_cepat_buah
     5     13             5_umkm_harga_usaha_plastik
     6     11       6_bahan_bahan baku_baku_industri
     7      7             7_bawa_tas_belanja_kantong
     8      7       8_sampah_kawasan_nasional_gunung
